In [4]:
import pandas as pd
import re

# --- CONFIGURATION: Input File Names ---
files = {
    'TSP_20': 'aggregated_tsp_n20_results.csv',
    'TSP_50': 'aggregated_tsp_n50_results.csv',
    'TSPTW_20': 'aggregated_tsptw_n20_results.csv',
    'TSPTW_50': 'aggregated_tsptw_n50_results.csv',
    'CVRP': 'aggregated_cvrp_results.csv',
    'CVRPTW': 'aggregated_cvrptw_results.csv',
    'JSP': 'aggregated_jsp_results.csv'
}

output_file = 'domain_aggregated_results.xlsx'

# --- HELPER FUNCTIONS ---

def clean_gap(val):
    """Convert string percentage (e.g., '5.64%') or float to float."""
    try:
        if isinstance(val, str):
            return float(val.strip().strip('%'))
        return float(val)
    except:
        return 0.0

def clean_optimality_val(val):
    """Normalize optimality values to boolean True/False."""
    s = str(val).lower().strip()
    if s == 'true': return True
    if s == 'false': return False
    return False # Default to False if ambiguous or Time Limit

def get_optimality_summary(series):
    """
    Returns True if all are True.
    Returns False if all are False.
    Returns None (Blank) if mixed.
    """
    vals = series.apply(clean_optimality_val).tolist()
    if all(vals):
        return True
    if not any(vals): # All False
        return False
    return None # Mixed

def process_subset(subset, label):
    """Calculates averages and summary stats for a filtered dataframe."""
    if len(subset) == 0:
        return None
        
    # Helper to safe-get columns from the input files
    def get_col(df, possible_names):
        for name in possible_names:
            if name in df.columns: return df[name]
        raise KeyError(f"Could not find any of {possible_names}")

    # Clean Gaps (remove % and convert to float)
    ea_gaps = get_col(subset, ['EA Dual Bound Optimality Gap']).apply(clean_gap)
    single_gaps = get_col(subset, ['Single Dual Bound Optimality Gap']).apply(clean_gap)
    
    # Get other columns
    ea_time = get_col(subset, ['EA Dual Bound Time'])
    ea_nodes = get_col(subset, ['EA Dual Bound Expanded Nodes'])
    ea_opt = get_col(subset, ['EA Dual Bound Optimality'])
    
    single_time = get_col(subset, ['Single Dual Bound Time'])
    single_nodes = get_col(subset, ['Single Dual Bound Expanded Nodes'])
    single_opt = get_col(subset, ['Single Dual Bound Optimality', 'Single Dual Bound Is Optimal'])

    # Return dictionary with Final Header Names
    return {
        'Instance Size': label,
        'Number of Instances': len(subset),  # <--- Added Count Column Here
        
        # EA Metrics
        'EA Dual Bound Average Optimality Gap (%)': round(ea_gaps.mean(), 2),
        'EA Dual Bound Average Time (s)': ea_time.mean(),
        'EA Dual Bound Optimality': get_optimality_summary(ea_opt),
        'EA Dual Bound Average Expanded Nodes': round(ea_nodes.mean()),
        
        # Single Metrics
        'Single Dual Bound Average Optimality Gap (%)': round(single_gaps.mean(), 2),
        'Single Dual Bound Average Time (s)': single_time.mean(),
        'Single Dual Bound Optimality': get_optimality_summary(single_opt),
        'Single Dual Bound Average Expanded Nodes': round(single_nodes.mean())
    }

# --- MAIN PROCESSING ---

results_data = {}

# 1. TSP
rows_tsp = []
df_tsp_20 = pd.read_csv(files['TSP_20'])
df_tsp_50 = pd.read_csv(files['TSP_50'])
rows_tsp.append(process_subset(df_tsp_20, 'n = 20'))
rows_tsp.append(process_subset(df_tsp_50, 'n = 50'))
results_data['TSP'] = pd.DataFrame(rows_tsp)

# 2. TSPTW
rows_tsptw = []
df_tsptw_20 = pd.read_csv(files['TSPTW_20'])
df_tsptw_50 = pd.read_csv(files['TSPTW_50'])
rows_tsptw.append(process_subset(df_tsptw_20, 'n = 20'))
rows_tsptw.append(process_subset(df_tsptw_50, 'n = 50'))
results_data['TSPTW'] = pd.DataFrame(rows_tsptw)

# 3. CVRP
df_cvrp = pd.read_csv(files['CVRP'])
df_cvrp['n_val'] = df_cvrp['Instance'].str.extract(r'-n(\d+)-').astype(float)
rows_cvrp = []
rows_cvrp.append(process_subset(df_cvrp[df_cvrp['n_val'] < 100], 'n < 100'))
rows_cvrp.append(process_subset(df_cvrp[df_cvrp['n_val'] >= 100], 'n >= 100'))
results_data['CVRP'] = pd.DataFrame([r for r in rows_cvrp if r])

# 4. CVRPTW
df_cvrptw = pd.read_csv(files['CVRPTW'])
def is_homberger_200(name):
    return '_2_' in str(name)

rows_cvrptw = []
rows_cvrptw.append(process_subset(df_cvrptw[~df_cvrptw['Instance'].apply(is_homberger_200)], 'n = 100'))
rows_cvrptw.append(process_subset(df_cvrptw[df_cvrptw['Instance'].apply(is_homberger_200)], 'n = 200'))
results_data['CVRPTW'] = pd.DataFrame([r for r in rows_cvrptw if r])

# 5. JSP (Updated Logic)
df_jsp = pd.read_csv(files['JSP'])

def get_jsp_category(name):
    name = str(name).lower().strip()
    
    # Explicit User Override for ft20 to be in "Large"
    if 'ft20' in name:
        return 'large'
    
    # Taillard (ta) instances are generally large (>150 ops)
    if name.startswith('ta'):
        return 'large'
        
    # Lawrence (la) instances
    # la01-la25 are <= 15x10 (150 ops) -> Small
    # la26-la40 are >= 20x10 (200 ops) -> Large
    if name.startswith('la'):
        try:
            num = int(re.findall(r'\d+', name)[0])
            if num <= 25:
                return 'small'
            else:
                return 'large'
        except:
            return 'large'
            
    # Standard small benchmarks (ft, abz, orb) usually <= 150 ops
    # (Except ft20 which was handled above)
    if name.startswith(('ft', 'abz', 'orb')):
        return 'small'
        
    return 'large' # Default

df_jsp['cat'] = df_jsp['Instance'].apply(get_jsp_category)
rows_jsp = []
rows_jsp.append(process_subset(df_jsp[df_jsp['cat'] == 'small'], 'n(operations) <= 150'))
rows_jsp.append(process_subset(df_jsp[df_jsp['cat'] == 'large'], 'n(operations) > 150'))
results_data['JSP'] = pd.DataFrame([r for r in rows_jsp if r])

# --- SAVE TO EXCEL ---
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    for sheet_name, df in results_data.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Formatting
        worksheet = writer.sheets[sheet_name]
        worksheet.set_column(0, 0, 30) # Instance Size column
        worksheet.set_column(1, 10, 25) # Data columns

print(f"Successfully created {output_file}")

Successfully created domain_aggregated_results.xlsx
